# 第43章：贝叶斯推断、因果推断与文本/大模型在量化中的实战指南

本 Notebook 涵盖三条完整的端到端教学与实验验证链：
1. **贝叶斯共轭更新与正态分层部分池化（Partial Pooling）**
2. **因果有向无环图（DAG）、辛普森悖论与标准化 ATE 及对撞偏差（Collider Bias）**
3. **替代数据时点（Point-in-Time, PIT）日历契约、120条事件（4固定模板循环）文本特征提取、无未来信息切分与 Ridge 预测基准**
4. **词嵌入矩阵、否定词情感评分与离线大模型提示词哈希摘要校验**


文本预测数值引用旧冻结summary，不重新拟合；无新市场证据。其余小型公式按原实现演算。

本 Notebook 使用离线教学构造数据，或明确标注的历史结果。新运行输出保存在单独的 `runs/` 子目录；这些内容不构成真实市场研究。


## 0. 环境设置与模块导入

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch43_bayesian_text_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
import json, hashlib
from quant_course.bayesian_text import (
    beta_binomial_posterior_predictive,
    normal_hierarchical_partial_pooling,
    generate_causal_dataset,
    compute_stratified_ate,
    demonstrate_collider_bias,
    generate_constructed_text_dataset,
    filter_pit_eligible_documents,
    hand_embedding_matrix,
    negation_sentiment_scorer,
    llm_offline_audit_ledger,
    downstream_execution_interface_spec,
)

CANDIDATE = ROOT / 'fixtures/late'
AUDIT = json.loads((CANDIDATE / 'display_audit.json').read_text())


## 1. 贝叶斯共轭推断与正态分层部分池化

### 1.1 Beta-Binomial 共轭更新
先验 $\text{Beta}(\alpha=2, \beta=2)$，观测到 10 次交易中有 8 次成功 ($s=8, n=10$)。
后验参数为 $\alpha' = 2+8=10, \beta' = 2+2=4$。
后验均值为 $\frac{10}{14} \approx 0.7143$，后验方差为 $\frac{40}{14^2 \times 15} = \frac{2}{147} \approx 0.0136$。
后验预测两连胜概率 $P(Y_1=1, Y_2=1 \mid D) = \frac{10 \times 11}{14 \times 15} = \frac{11}{21} \approx 0.5238$。
后验预测协方差 $\operatorname{Cov}(Y_1, Y_2 \mid D) = \operatorname{Var}(p \mid D) = \frac{2}{147} \approx 0.0136$。

In [ ]:
bayes_res = beta_binomial_posterior_predictive(alpha_prior=2.0, beta_prior=2.0, successes=8, trials=10)
print(f"Alpha Posterior: {bayes_res['alpha_post']}")
print(f"Beta Posterior: {bayes_res['beta_post']}")
print(f"Posterior Mean: {bayes_res['posterior_mean']:.4f}")
print(f"Posterior Variance: {bayes_res['posterior_variance']:.6f}")
print(f"Joint 2 Successes Predictive: {bayes_res['joint_two_successes']:.4f}")
print(f"Predictive Covariance: {bayes_res['predictive_covariance']:.6f}")

### 1.2 正态分层部分池化 (Normal Partial Pooling with Known Hyperparameters)
对于跨资产收益率估计，设定总体均值 $\mu=0.0$，总体异质性标准差 $\tau=0.02$，观测噪声 $\sigma=0.04$。
组 1 ($n_1=4, \bar{y}_1=0.03$): 收缩权重 $w_1 = \frac{4}{4+4} = 0.5$，后验均值 $\hat{\theta}_1 = 0.015$。
组 2 ($n_2=36, \bar{y}_2=0.01$): 收缩权重 $w_2 = \frac{36}{36+4} = 0.9$，后验均值 $\hat{\theta}_2 = 0.009$。

In [ ]:
groups_spec = [{'n': 4, 'sample_mean': 0.03}, {'n': 36, 'sample_mean': 0.01}]
normal_res = normal_hierarchical_partial_pooling(
    prior_mean=0.0, prior_sd=0.02, observation_sd=0.04, groups=groups_spec
)
for g in normal_res['groups']:
    print(f"Group {g['group_id']}: n={g['n']}, sample_mean={g['sample_mean']:.3f}, "
          f"shrinkage_weight={g['shrinkage_weight']:.2f}, post_mean={g['posterior_mean']:.4f}, "
          f"post_var={g['posterior_var']:.6f}, pred_var={g['predictive_var']:.6f}")

### 1.3 贝叶斯共轭更新与分层收缩示意图

In [ ]:
fig1_path = CANDIDATE / 'ch43' / 'fig43_1_bayesian_updating.png'
display(Image(filename=str(fig1_path)))

## 2. 因果推断、混杂控制与辛普森悖论

### 2.1 辛普森悖论反转与标准化 ATE
混杂变量 $Z$ 为分析师关注度（High vs Low）。未调整朴素差异显示 $+0.38$（38 个百分点），
但在两个层内真实处理效应均为 $-0.10$（-10 个百分点）。
标准化 ATE 消除后门路径混杂后得到正确的因果效应 $-0.10$。

In [ ]:
causal_df = generate_causal_dataset(seed=4301)
causal_res = compute_stratified_ate(
    causal_df, z_col='Z', t_col='T', y_col='Y', target_p_z={'high': 0.5, 'low': 0.5}
)
print(f"Unadjusted Difference: {causal_res['unadjusted_difference']:+.2f}")
print(f"High Stratum Contrast: {causal_res['stratum_stats']['high']['contrast']:+.2f}")
print(f"Low Stratum Contrast:  {causal_res['stratum_stats']['low']['contrast']:+.2f}")
print(f"Standardized ATE:      {causal_res['standardized_ate']:+.2f}")
print(f"Simpson Reversal Detected: {causal_res['simpson_reversal']}")

### 2.2 对撞偏差（Collider Bias）数值仿真
当以两个独立因子（动量 $T$ 与质量 $Y$）的联合准入条件 $C = \mathbb{I}(T=1 \lor Y=1)$ 筛选持仓时，
对对撞结构 $C$ 进行条件限制会产生虚假的负相关 $\operatorname{Cov}(T, Y \mid C=1) = -1/9 \approx -0.1111$。

In [ ]:
collider_res = demonstrate_collider_bias(n=2000, seed=4301)
print(f"Unconditioned Covariance: {collider_res['unconditioned_covariance']:+.4f}")
print(f"Conditioned on Collider Covariance: {collider_res['conditioned_on_collider_covariance']:+.4f}")
print(f"Theoretical Covariance: {-1.0/9.0:+.4f}")

### 2.3 辛普森效应与因果 DAG 示意图

In [ ]:
fig2_path = CANDIDATE / 'ch43' / 'fig43_2_causal_simpson.png'
display(Image(filename=str(fig2_path)))

fig5_path = CANDIDATE / 'ch43' / 'fig43_5_causal_dag.png'
display(Image(filename=str(fig5_path)))

## 3. 时点（PIT）日历契约与文本特征预测基准

### 3.1 构造的 120条事件（4固定模板循环）业绩电话会简报与 PIT 筛选
- 文本 `source_id` 显式标注 `CONSTRUCTED_CALL_`，明确教学构造。
- 盘后发布（16:05 NY）、供应商分发延迟（16:25 NY）、内部入库（16:30 NY）。
- 信号生成时间为次日 09:00 NY，严格满足内部入库早于信号时点。
- 训练集（59 篇）、验证集（29 篇）、测试集（30 篇），严格依据标签成熟时点早于后续信号进行无泄漏划分。

120事件仅4固定模板循环，59/29/30成熟块各含同4模板，只验证时间块外推，未验证新文本泛化。

In [ ]:
base_text_df = pd.read_csv(CANDIDATE/'ch43/constructed_documents.csv')
for name in ['event_time','release_time','available_at','ingested_at','revision_time','signal_time','trade_time','label_start','label_end','label_available_at']:
 if name in base_text_df:base_text_df[name]=pd.to_datetime(base_text_df[name],utc=True)
eligible_df=filter_pit_eligible_documents(base_text_df,cutoff_col='signal_time')
assert len(eligible_df)==120
print(AUDIT['t43_01'])

### 3.2 负例检验：晚到修订与未入库版本隔离
验证晚于信号时点的修订版本和入库延迟记录被严格拒绝并剔除。

In [ ]:
late_rev=base_text_df.iloc[10:11].copy()
late_rev['version_id']=2
late_rev['revision_time']=late_rev.signal_time+pd.Timedelta(hours=1)
late_rev['available_at']=late_rev.revision_time
late_rev['ingested_at']=late_rev.available_at+pd.Timedelta(minutes=5)
filtered=filter_pit_eligible_documents(pd.concat([base_text_df,late_rev],ignore_index=True),cutoff_col='signal_time')
assert len(filtered)==120
assert int(filtered.loc[filtered.doc_id==10,'version_id'].iloc[0])==1
print('Late-revision small negative control retained; no fit.')

### 3.3 严格时序划分与 TF-IDF Ridge 预测实验
比较训练均值基准与 Ridge 回归（超参数 $\alpha \in [0.1, 1.0, 10.0]$）。
计算测试集 MSE、$R^2$ 与跨 30 个测试交易日的时序汇总 Spearman 秩相关系数。
明确说明：因每日仅有单个资产，截面 IC 未定义；常数均值基准秩相关未定义。

以下仅显示旧冻结预测估计；OOV词表/IDF不变不等于新文本能力。

In [ ]:
text_res=json.loads((CANDIDATE/'ch43/ch43_summary.json').read_text())['text_nlp']
display(pd.Series({k:v for k,v in text_res.items() if isinstance(v,(str,int,float))}))
assert text_res['n_train_eligible']==59 and text_res['n_val_eligible']==29 and text_res['n_test']==30
print('Frozen prediction estimates, not fitted in this revision.')

### 3.4 时点契约时序图与测试评估指标对比图

In [ ]:
fig3_path = CANDIDATE / 'ch43' / 'fig43_3_pit_text_timeline.png'
display(Image(filename=str(fig3_path)))

fig4_path = CANDIDATE / 'ch43' / 'fig43_4_text_prediction_metrics.png'
display(Image(filename=str(fig4_path)))

## 4. 密集词嵌入与否定词情感评分

### 4.1 教学型 6x3 稠密嵌入矩阵与余弦相似度
无需下载庞大外部模型，以稠密向量展示语义几何结构与均值池化（Mean Pooling）。

In [ ]:
df_emb, sims = hand_embedding_matrix()
print("Normalized Embedding Matrix:")
print(df_emb.round(3))
print("\nCosine Similarities:")
for k, v in sims.items():
    print(f"{k}: {v:+.4f}")

### 4.2 否定词极性反转实验
对比朴素词袋（Bag-of-Words）与考虑否定词窗口的情感评分。

In [ ]:
sentence = 'earnings performance did not improve showing no growth despite aggressive restructuring'
score_res = negation_sentiment_scorer(sentence)
print(f"Input Text: {score_res['text']}")
print(f"Naive Unigram Score:       {score_res['naive_unigram_score']} (misclassifies positive words)")
print(f"Negation-Aware Score:      {score_res['negation_aware_score']} (correctly flips polarity)")
print(f"Polarity Flip Detected:    {score_res['polarities_differ']}")

## 5. 离线大模型提示词哈希摘要校验与下游接口规约

### 5.1 离线大模型研究哈希审计台账
- 模型调用状态标注为 `not_run`，模型截断日标注为未知（假设反例为 2023-12-31）。
- 计算包含真实输入文本的完整提示词 SHA-256 哈希值。
- 强调预训练知识截断日不等于时点可核资产记录，零温与种子不能消除跨平台漂移。

密码学哈希是摘要一致性校验，需要独立可信预期摘要。不是加密/不可变存证；输入与摘要一起改变仍能通过本地重算。本例演示比较，不提供可信保管机制。

In [ ]:
ledger = llm_offline_audit_ledger()
print(f"Prompt Template Version: {ledger['prompt_template_version']}")
print(f"Full Prompt SHA-256:     {ledger['full_prompt_sha256_hash']}")
print(f"Model Status:            {ledger['model_status']}")
print(f"Declared Model Cutoff:   {ledger['declared_model_cutoff']}")
print(f"Provenance Table:        {ledger['provenance_table']}")
print("Reproducibility Limits:")
for lim in ledger['reproducibility_limits']:
    print(f" - {lim}")
# Assume the baseline digest was independently retained before local mutation.
from quant_course.bayesian_text import compute_prompt_hash, OFFLINE_PROMPT_TEMPLATE
trusted_digest=ledger['full_prompt_sha256_hash']
changed=llm_offline_audit_ledger(sample_text='changed input and locally matching hash')
assert compute_prompt_hash(OFFLINE_PROMPT_TEMPLATE,changed['sample_input_text'])==changed['full_prompt_sha256_hash']
assert changed['full_prompt_sha256_hash']!=trusted_digest
print('Local consistency passes after input+digest mutation; independent baseline differs.')


### 5.2 下游执行接口规约（Prediction-Only）
本章仅负责时点合规的预测信号产出，不虚构轮换资产的成交账本。
下游执行系统须接入标准持仓权重与费用账本接口。

In [ ]:
downstream_spec = downstream_execution_interface_spec()
print(f"Mode: {downstream_spec['mode']}")
print(f"Required Schema: {downstream_spec['required_schema']}")
print(f"Scope Boundary: {downstream_spec['scope_boundary']}")